# get the data :

In [4]:
import os
import json
import itertools
import csv
from google.colab import drive


# Reconnecter Drive proprement
try:
    drive.flush_and_unmount()
except:
    pass
drive.mount('/content/drive', force_remount=True)


# --- CONFIGURATION ---
DATA_DIRS = [
    "/content/drive/.shortcut-targets-by-id/14T7-Rrikc5faA88C46ODGJuh9gw_rH00/Abderahmane"
    # "/content/drive/.shortcut-targets-by-id/1p-Puanbb3bPpRFF5xKN2a4vLn-G7eupa/",
    # "/content/drive/.shortcut-targets-by-id/1GCz66KY_YAruf3NxHOOtMw5H3ykjQRN3/Bilel",
    # "/content/drive/.shortcut-targets-by-id/1fwW853am4kHWdNXr3yuOe-q-P2LTDnOW/Rayane",
    # "/content/drive/.shortcut-targets-by-id/1Lh4jJesDGb9sWrUb0Hlg6a847oMsiPj4/Zohra"

]


for folder in DATA_DIRS:
    if not os.path.exists(folder):
        print(f"{folder} → Folder not found")
        continue

    count = 0
    for _, _, files in os.walk(folder):
        count += len(files)

    print(f"{folder} →  {count} files")

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/14T7-Rrikc5faA88C46ODGJuh9gw_rH00/Abderahmane →  235 files


# Filter

In [5]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=695d283662d9544660098b9a514afe5ad70a1ce940ec72088f4e8fc85efdf78b
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [6]:
from langdetect import detect, LangDetectException

In [ ]:
OUTPUT_FILE = "/content/drive/MyDrive/nlp data/scopus_english_ids_abdrahmane.txt"

def load_articles(file_path):
    """Load articles no matter the JSON format."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, list):
            return data
        if isinstance(data, dict) and "articles" in data:
            return data["articles"]
        return []
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []


def check_scopus(article):
    """Safely check if article is indexed in Scopus."""
    try:
        return article["primary_location"]["source"]["is_indexed_in_scopus"]
    except:
        return False


def reconstruct_abstract(abstract_inverted_index):
    """Reconstruct abstract text from inverted index."""
    if not abstract_inverted_index or not isinstance(abstract_inverted_index, dict):
        return ""

    word_positions = []
    for word, positions in abstract_inverted_index.items():
        if isinstance(positions, list):
            for pos in positions:
                word_positions.append((pos, word))

    word_positions.sort(key=lambda x: x[0])
    return " ".join([word for pos, word in word_positions])


def is_english(text):
    """
    Detect if text is in English (not French, Spanish, German, Chinese, Russian, etc.)
    """
    if not text or len(text.strip()) == 0:
        return False

    # Need at least ~20 characters for reliable detection
    if len(text.strip()) < 20:
        # For very short text, fallback to Latin char check
        cleaned = ''.join(c for c in text if c.isalpha())
        if len(cleaned) == 0:
            return False
        latin_chars = sum(1 for c in cleaned if ord(c) < 128 and c.isalpha())
        return (latin_chars / len(cleaned)) >= 0.90

    try:
        detected_lang = detect(text)
        return detected_lang == 'en'
    except LangDetectException:
        # If detection fails, fallback to False (reject uncertain cases)
        return False


def is_article_english(article):
    """Check if both title and abstract are in English."""
    title = article.get("title", "") or article.get("display_name", "")
    if not is_english(title):
        return False

    abstract_inverted = article.get("abstract_inverted_index")
    if abstract_inverted:
        abstract = reconstruct_abstract(abstract_inverted)
        if not is_english(abstract):
            return False

    return True


# ---- MAIN PROCESSING WITH IMMEDIATE WRITING ----
total_files = 0
total_articles = 0
scopus_articles = 0
non_english_filtered = 0
english_scopus_count = 0

print("Starting to process directories...")
print(f"Output file: {OUTPUT_FILE}")
print("Writing results in real-time (crash-proof)...\n")

#  Open file ONCE at the start, keep it open during processing
with open(OUTPUT_FILE, "w", encoding="utf-8") as output_file:

    for data_dir in DATA_DIRS:
        if not os.path.exists(data_dir):
            print(f"Directory not found: {data_dir}")
            continue

        print(f"\nProcessing: {data_dir}")

        for root, dirs, files in os.walk(data_dir):
            for file in files:
                if file.endswith(".json"):
                    file_path = os.path.join(root, file)
                    total_files += 1

                    articles = load_articles(file_path)
                    total_articles += len(articles)

                    for article in articles:
                        if check_scopus(article):
                            scopus_articles += 1

                            if is_article_english(article):
                                article_id = article.get("id")
                                if article_id:
                                    # Write IMMEDIATELY to file
                                    output_file.write(article_id + "\n")
                                    output_file.flush()  #  Force write to disk NOW
                                    english_scopus_count += 1
                            else:
                                non_english_filtered += 1

                    # Free memory
                    articles = None

                    if total_files % 5 == 0:
                        print(f"  Files: {total_files} | English Scopus: {english_scopus_count} | Non-English: {non_english_filtered}")

# Summary
print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Total JSON files processed: {total_files}")
print(f"Total articles found: {total_articles}")
print(f"Scopus-indexed articles: {scopus_articles}")
print(f"Non-English filtered out: {non_english_filtered}")
print(f"Final English Scopus articles: {english_scopus_count}")
print(f"Output file: {OUTPUT_FILE}")
print("="*60)

Starting to process directories...
Output file: /content/drive/MyDrive/nlp data/scopus_english_ids_abdrahmane.txt
Writing results in real-time (crash-proof)...


Processing: /content/drive/.shortcut-targets-by-id/14T7-Rrikc5faA88C46ODGJuh9gw_rH00/Abderahmane
  Files: 5 | English Scopus: 21389 | Non-English: 284
  Files: 10 | English Scopus: 43006 | Non-English: 828
  Files: 15 | English Scopus: 65146 | Non-English: 1360
  Files: 20 | English Scopus: 87382 | Non-English: 1798
  Files: 25 | English Scopus: 109660 | Non-English: 2142
